# Repair Script

Fixes incorrect summaries afflicted by these three conditions:
1. Truncated due to previous token limits (320, 1000, and 10000 at different times)
2. Original chunk text retained instead of summary due to previous rule that original chunk would be retained if summary length > original chunk length (current softer fix = this guideline is incorporated into the prompt) [if <= 8 words then it's fine because a summary is useless at that length]
3. Table summaries would include tabular data in the summary, so I set this to regenerate EVERY table summary

Using a cache to retain work even if notebook crashes due to kernel restart, dropped SSH, etc.

In [1]:
%pip install ollama tiktoken

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import os, json, re, time, hashlib, textwrap
from pathlib import Path
from dataclasses import dataclass, field
from typing import List, Dict, Any, Optional

import ollama
import tiktoken

# models
OLLAMA_URL    = "http://localhost:11528"
SUMMARY_MODEL = "gpt-oss:120b"      # text/tables/combine summaries
VISION_MODEL  = "gemma3:27b"        # figure descriptions

# paths
CACHE_DIR      = Path("tree_cache")
TREE_FILE      = CACHE_DIR / "corpus_tree.json"
NODE_CACHE_DIR = CACHE_DIR / "nodes"
IMG_DIR        = CACHE_DIR / "images"
REPORT_FILE    = CACHE_DIR / "retained_chunks_report.txt"

# test configs
DRY_RUN          = False   
REGEN_IMAGES     = True   
UPDATE_NODE_CACHE = True   
BACKUP_TREE      = True    

# --- crash-resume -----------------------------------------------------------
# Every repaired summary is appended to REPAIR_LOG (fsync'd) the instant it's
# made. On restart the log is replayed onto the tree and finished nodes are
# skipped, so a crash never sends you back to 0. CHECKPOINT_EVERY also rewrites
# the full tree periodically so corpus_tree.json reflects progress mid-run.
REPAIR_LOG       = CACHE_DIR / "repaired_summaries.jsonl"
CHECKPOINT_EVERY = 2000    # rewrite the full tree every N fixes (0 = only at the end)
SHOW_SUMMARIES   = False   # per-node summary printing off (keeps long runs light)

# A leaf whose summary == its content was retained verbatim. Those with <= this
# many words were genuinely too short to summarise (fine, listed in the report).
# Those with MORE words were retained only because the old build discarded a real
# summary for being longer than the source; with that guard now gone, re-summarise
# them when RESUMMARIZE_LONG_VERBATIM is on.
SKIP_SUMMARY_WORDS       = 8     # must match the build's short-skip threshold
RESUMMARIZE_LONG_VERBATIM = True  # re-summarise verbatim chunks longer than that
REGEN_ALL_TABLES         = True  # regenerate EVERY table summary with the updated prompt

# 320, 1000, and 10000 tokens were the previous summary limit configurations
TRUNCATION_CAPS    = [320, 1000, 10000]
CAP_MARGIN         = 12      # band half-width tokens around each cap
ALSO_FLAG_NO_PUNCT = True    # also flag summaries that don't end in . ! ?

# tiktoken for counting 320/1000/10000 tokents which were the previous limits resulting in truncation
_ENC = None
for _enc_name in ("o200k_base", "cl100k_base"):
    try:
        _ENC = tiktoken.get_encoding(_enc_name)
        break
    except Exception:
        continue
if _ENC is not None:
    def _count_tokens(s: str) -> int:
        return len(_ENC.encode(s)) if s else 0
    print(f"Token counter: tiktoken/{_ENC.name}")
else:
    CAP_MARGIN = max(CAP_MARGIN, 40)
    _WORDRE = re.compile(r"\S+")
    def _count_tokens(s: str) -> int:                    # ~1.33 tokens/word
        return round(len(_WORDRE.findall(s)) * 1.33) if s else 0
    print("tiktoken vocab could not be loaded (offline?). Using a word-based token "
          f"estimate; widened CAP_MARGIN to {CAP_MARGIN}.")

GEN_NUM_PREDICT  = 1000000  # uncapped so regenerated summaries finish w/o truncation
KEEP_ALIVE       = "30m"
DISABLE_THINKING = True     # gpt-oss reasoning off for speed during bulk regeneration
RETRY_WAIT_MAX   = 60       # wait-and-retry if Ollama disconnects
READY_POLL       = 10

print(f"Tree   : {TREE_FILE}")
print(f"Models : text={SUMMARY_MODEL}  vision={VISION_MODEL} (regen_images={REGEN_IMAGES})")
print(f"Mode   : {'DRY RUN (no changes)' if DRY_RUN else 'REPAIR (will rewrite summaries)'}")

Token counter: tiktoken/o200k_base
Tree   : tree_cache/corpus_tree.json
Models : text=gpt-oss:120b  vision=gemma3:27b (regen_images=True)
Mode   : REPAIR (will rewrite summaries)


In [ ]:
OLLAMA_TIMEOUT = 600  # per-request timeout in seconds; generous so a slow regeneration call doesn't get cut off
client = ollama.Client(host=OLLAMA_URL, timeout=OLLAMA_TIMEOUT)  # the shared client every call goes through, pointed at the tunnel
_THINK = {"use": DISABLE_THINKING}  # mutable flag for whether to send think=false, in a dict so the helpers can flip it


def _model_names(r):
    # pulls the plain list of model name strings out of client.list(), coping with whichever response shape the ollama version returns
    raw = r.get("models", []) if hasattr(r, "get") else getattr(r, "models", [])
    out = []
    for m in raw:
        n = getattr(m, "model", None) or getattr(m, "name", None)
        if n is None and isinstance(m, dict):
            n = m.get("model") or m.get("name")
        if n:
            out.append(n)
    return out


def _wait_backoff(a):
    # sleeps a bit longer each retry, 5s 10s 20s 40s 80s but capped, so we don't hammer the server while it's down
    time.sleep(min(RETRY_WAIT_MAX, 5 * (2 ** min(a - 1, 4))))


def _wait_until_ready():
    # waits for the server and the models the repair needs to be up, only requiring the vision model when we're regenerating images, looping forever rather than crashing if things aren't ready
    need = [SUMMARY_MODEL] + ([VISION_MODEL] if REGEN_IMAGES else [])
    announced = False
    while True:
        try:
            names = _model_names(client.list())
            missing = [m for m in need if not any(m in n for n in names)]
            if not missing:
                print(f"Ollama OK — models present: {', '.join(need)}")
                return
            reason = f"model(s) not loaded yet: {missing}"
        except Exception as e:
            reason = f"server unreachable ({type(e).__name__}: {e})"
        if not announced:
            print(f"Waiting for Ollama — {reason}. Re-checking every {READY_POLL}s; won't stop.")
            announced = True
        time.sleep(READY_POLL)


def _warm(model):
    # nudges a model into memory with a tiny throwaway call before a phase, so the first real regeneration isn't stuck on a cold load
    try:
        client.chat(model=model, messages=[{"role": "user", "content": "ok"}],
                    options={"num_predict": 1}, keep_alive=KEEP_ALIVE)
    except Exception:
        pass


def _chat(messages):
    # the resilient text call, sends to the summary model and waits-and-retries forever if Ollama drops instead of dying mid-repair
    opts = {"temperature": 0, "num_predict": GEN_NUM_PREDICT}
    attempt = 0
    while True:
        kw = dict(model=SUMMARY_MODEL, messages=messages, options=opts, keep_alive=KEEP_ALIVE)
        if _THINK["use"]:
            kw["think"] = False
        try:
            return client.chat(**kw)
        except TypeError:
            _THINK["use"] = False
        except Exception as e:
            if _THINK["use"]:
                _THINK["use"] = False
                continue
            attempt += 1
            if attempt == 1 or attempt % 5 == 0:
                print(f"[waiting for Ollama] {type(e).__name__}: {e} — retrying...")
            _wait_backoff(attempt)


def _words(s):
    # splits text into words, every run of non-whitespace, empty list if there's nothing
    return re.findall(r"\S+", s or "")


def _llm_summarise(text, context_hint=""):
    # re-summarises one text or table unit, and unlike the build notebook it deliberately keeps whatever summary the model produces even if it ends up longer, because the whole point here is fixing truncated summaries, not falling back to raw text
    if not text.strip():
        return "(empty)"
    hint = f" The content comes from: {context_hint}." if context_hint else ""
    prompt = (f"Summarise the following text.{hint} Focus on key topics, concepts, and "
              "specific information (names, numbers, procedures, entities). If the content "
              "is a Markdown table, describe what the table is for — what it tabulates and "
              "what its columns and rows represent — so a reader knows what they would find "
              "in it. Do NOT reproduce the table or list its cell values row by row; "
              "summarise what the table does, not its contents. "
              "Your summary should be at most 200 words and ideally shorter than the source, "
              "but if it cannot be shorter, still return your summary — never return the "
              "original text verbatim. Respond with ONLY the summary.\n\n" + text)
    # NOTE: the summary is returned as-is. We do NOT fall back to the source text
    # when the summary is longer — producing a real summary is the point.
    return _chat([{"role": "user", "content": prompt}])["message"]["content"].strip()


def _llm_combine_summaries(summaries, label):
    # re-combines child summaries into a parent summary, same keep-the-summary stance as above so a repaired parent reflects its corrected children
    if not summaries:
        return "(no content)"
    joined = "\n".join(f"- {s}" for s in summaries)
    prompt = (f"You are summarising a section, document, or folder called '{label}'. Below are "
              "summaries of its contents. Write ONE specific summary of the overall scope and key "
              "topics. It should be at most 200 words and ideally shorter than the combined input, "
              "but if it cannot be shorter, still return your summary — never return the input "
              "verbatim. Respond with ONLY the summary.\n\n" + joined)
    return _chat([{"role": "user", "content": prompt}])["message"]["content"].strip()


def _resolve_image(image_path):
    # tracks down a figure file on disk, trying the path as given, then the image folder, then a recursive search as a last resort
    if not image_path:
        return None
    p = Path(image_path)
    if p.exists():
        return p
    cand = IMG_DIR / p.name
    if cand.exists():
        return cand
    hits = list(IMG_DIR.rglob(p.name))
    return hits[0] if hits else None


def _llm_describe_image(image_path, caption_hint=""):
    # re-describes a figure with the vision model, and like _chat it waits and retries forever rather than giving up if the call fails
    p = _resolve_image(image_path)
    if p is None:
        return f"(figure; image file not found){(' Caption: ' + caption_hint) if caption_hint else ''}"
    hint = f" The figure's caption is: {caption_hint}." if caption_hint else ""
    prompt = ("Describe this figure from a document in 3-5 sentences so it can be found by "
              f"search.{hint} State the kind of visual, what it depicts, any axis labels / units / "
              "labelled parts you can read, and the main takeaway. Respond with ONLY the description.")
    attempt = 0
    while True:
        try:
            return client.chat(model=VISION_MODEL,
                               messages=[{"role": "user", "content": prompt, "images": [str(p)]}],
                               options={"temperature": 0, "num_predict": GEN_NUM_PREDICT},
                               keep_alive=KEEP_ALIVE)["message"]["content"].strip()
        except Exception as e:
            attempt += 1
            if attempt == 1 or attempt % 5 == 0:
                print(f"[waiting for Ollama/vision] {type(e).__name__}: {e} — retrying...")
            _wait_backoff(attempt)


@dataclass  # auto-generate the boilerplate from the fields below
class TreeNode:  # the node type, with to_dict here as well as from_dict since the repair writes the tree back out
    node_id: str; node_type: str; name: str; path: str; summary: str
    content: str = ""
    children: List["TreeNode"] = field(default_factory=list)
    metadata: Dict[str, Any] = field(default_factory=dict)
    def to_dict(self):
        # serialises this node and its subtree back to plain dicts, needed because this notebook saves the repaired tree
        return {"node_id": self.node_id, "node_type": self.node_type, "name": self.name,
                "path": self.path, "summary": self.summary, "content": self.content,
                "children": [c.to_dict() for c in self.children], "metadata": self.metadata}
    @classmethod
    def from_dict(cls, d):
        # rebuilds a node and its whole subtree from saved json, tolerating missing optional fields
        n = cls(d["node_id"], d["node_type"], d["name"], d.get("path", ""), d.get("summary", ""),
                d.get("content", ""), metadata=d.get("metadata", {}))
        n.children = [cls.from_dict(c) for c in d.get("children", [])]
        return n


def _node_cache_path(nid):
    # builds the on-disk path for a node's cache file, one json per node id
    return NODE_CACHE_DIR / f"{nid}.json"


def _save_cached_node(node):
    # writes a node to the cache atomically, temp file then rename, so a repaired node can't be left half-written if something dies mid-write
    NODE_CACHE_DIR.mkdir(parents=True, exist_ok=True)
    p = _node_cache_path(node.node_id); tmp = p.with_suffix(".tmp")
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(node.to_dict(), f, ensure_ascii=False)
    tmp.replace(p)


def _fmt_eta(s):
    # turns seconds into a tidy clock string like 1:02:03 or 4:05, dropping the hours when there aren't any
    s = int(max(0, s)); h, r = divmod(s, 3600); m, sec = divmod(r, 60)
    return f"{h}:{m:02d}:{sec:02d}" if h else f"{m}:{sec:02d}"


class Progress:  # the single progress bar for the repair pass, same idea as the build notebook's but counting fixes
    def __init__(self, total, desc="Repairing"):
        # sets up the bar over the number of repairs expected, recording the start time for rate and ETA
        from tqdm.auto import tqdm
        self.total = max(0, total); self.done = 0; self.t0 = time.time()
        self.bar = tqdm(total=self.total, desc=desc, unit="fix", dynamic_ncols=True)
    def set_phase(self, label):
        # relabels the bar to show which phase of the repair is running
        self.bar.set_description(label); self.bar.refresh()
    def step(self, label, dt, summary=None):
        # records one finished fix, ticks the bar, updates ETA and rate, and optionally prints the regenerated summary so you can eyeball it
        self.done += 1; self.bar.update(1)
        el = time.time() - self.t0; rate = self.done / el if el > 0 else 0
        rem = (self.total - self.done) / rate if rate > 0 else 0
        self.bar.set_postfix_str(f"ETA {_fmt_eta(rem)} | {rate:.2f}/s | last {dt:.1f}s")
        if summary is not None and SHOW_SUMMARIES:
            self.bar.write(f"== {label} ==")
            for line in textwrap.wrap(re.sub(r'\s+', ' ', summary).strip(), width=100):
                self.bar.write("  " + line)
            self.bar.write("")
    def close(self):
        # snaps the total to however many fixes actually happened and shuts the bar down
        self.bar.total = self.done; self.bar.refresh(); self.bar.close()


if not DRY_RUN:
    _wait_until_ready()  # only bother waiting on the server when we're actually going to regenerate; a dry run just classifies and needs no model
print("Helpers ready (resilient regeneration + cache + progress)")

Ollama OK — models present: gpt-oss:120b, gemma3:27b
Helpers ready (resilient regeneration + cache + progress)


In [ ]:
if not TREE_FILE.exists():
    # bail out early with a clear message if the tree file isn't there, since there's nothing to audit or repair without it
    raise SystemExit(f"Tree not found: {TREE_FILE.resolve()} — run the build (prototype 5) first.")
with open(TREE_FILE, encoding="utf-8") as f:
    ROOT = TreeNode.from_dict(json.load(f))  # load the saved json and rebuild the whole tree into ROOT, the tree this notebook will scan and fix
def _count(n):
    # counts every node in a subtree including itself, recursing through children, just for the stats line below
    return 1 + sum(_count(c) for c in n.children)
print(f"Loaded tree: {ROOT.name} | {_count(ROOT)} nodes")  # confirm the load worked and show the root name and total node count

Loaded tree: folders | 130154 nodes


In [ ]:
def _norm(s: str) -> str:
    # collapses whitespace to single spaces and trims, so comparisons and length checks aren't thrown off by formatting
    return re.sub(r"\s+", " ", s or "").strip()


_TRAIL = "\"'`’”)]}»…"  # trailing quotes, brackets, and ellipsis to peel off before checking how a summary ends


def _is_placeholder(s: str) -> bool:
    # tells whether a summary is one of the failure placeholders left by a previous run, so we treat those as skipped rather than truncated
    t = (s or "").strip()
    if t in ("", "(empty)", "(no content)"):
        return True
    return t.startswith(("(summary unavailable", "(image description unavailable",
                         "(figure; image file not found", "(error", "(skipped logo"))


def _ends_complete(s: str) -> bool:
    # decides whether a summary ends on a real sentence boundary, treating a trailing "..." as a sign it was cut off mid-thought
    t = _norm(s).rstrip(_TRAIL).rstrip()
    if not t:
        return False
    if t.endswith("..."):                 # explicit ellipsis = cut off
        return False
    return t[-1] in ".!?"


def _truncation_reason(s: str):
    """Why this summary looks truncated, or None. Primary signal: its tiktoken
    length sits in a band around one of the generation caps the build used
    (TRUNCATION_CAPS) — a response cut off by the limit lands right at that cap.
    Secondary signal (union, to avoid false negatives e.g. when reasoning tokens
    ate into a gpt-oss budget): it doesn't end in sentence punctuation."""
    # the heart of the audit, it flags a summary as truncated mainly when its token count lands right on one of the old generation caps, since that's where a cut-off response stops, and as a backup also flags one that doesn't end in punctuation, the union being deliberate so we err toward catching truncations rather than missing them
    n = _count_tokens(s)
    for c in TRUNCATION_CAPS:
        if c - CAP_MARGIN <= n <= c + CAP_MARGIN:
            return f"~{c} tokens (got {n})"
    if ALSO_FLAG_NO_PUNCT and not _ends_complete(s):
        return "no end punctuation"
    return None


def _is_verbatim(node) -> bool:
    # detects a leaf whose summary is just a copy of its content, which is how a short chunk gets stored, and the marker for the verbatim cases we sort out below
    return (node.node_type == "chunk" and node.content.strip() != ""
            and _norm(node.summary) == _norm(node.content))


def classify_tree(root):
    """Return dict of lists. case2 items are (node, depth)."""
    # the sorting pass, it walks the whole tree and buckets every node into complete, skipped, genuinely-verbatim, or "case2" which means needs regenerating, and tallies the reasons, with a couple of switches, force-regenerate every table to pick up the better table prompt, and re-summarise long verbatim chunks that were really just casualties of the old shorter-than-source guard
    from collections import Counter
    out = {"complete": 0, "skipped": 0, "verbatim": [], "case2": [],
           "reasons": Counter()}

    def walk(node, depth):
        # the recursive worker that classifies one node and then recurses into its children, carrying depth so repair can later go deepest-first
        s = node.summary or ""
        is_table = (node.node_type == "chunk" and node.metadata.get("kind") == "table")
        if REGEN_ALL_TABLES and is_table:
            out["case2"].append((node, depth))         # force-regenerate EVERY table
            out["reasons"]["table -> regenerate (all)"] += 1
        elif _is_verbatim(node):
            wc = len(_norm(node.content).split())
            if RESUMMARIZE_LONG_VERBATIM and wc > SKIP_SUMMARY_WORDS:
                out["case2"].append((node, depth))    # guard casualty -> re-summarise
                out["reasons"][f"verbatim >{SKIP_SUMMARY_WORDS}w -> re-summarise"] += 1
            else:
                out["verbatim"].append(node)           # genuinely short -> retained (fine)
        elif node.metadata.get("skipped") or _is_placeholder(s):
            out["skipped"] += 1
        else:
            reason = _truncation_reason(s)
            if reason:
                out["case2"].append((node, depth))     # TRUNCATED -> regenerate
                out["reasons"][reason.split(" (")[0]] += 1
            else:
                out["complete"] += 1
        for c in node.children:
            walk(c, depth + 1)

    walk(root, 0)
    return out


def _regen_node(node):
    """Regenerate one truncated node's summary IN PLACE. Returns a status string."""
    # actually rebuilds one node's summary in place, describing the figure for an image leaf, re-summarising the text or table for other leaves, and re-combining from its children for a section, document, or folder, and returns a tag saying which it did
    nt = node.node_type
    if nt == "chunk":
        kind = node.metadata.get("kind", "text")
        if kind == "image":
            ip = node.metadata.get("image_path", "")
            cap = node.metadata.get("caption", "")
            if _resolve_image(ip) is None:
                return "skip_no_image"
            desc = _llm_describe_image(ip, caption_hint=cap)
            node.summary = desc
            node.content = ("[FIGURE] " + (cap + " — " if cap else "") + desc).strip()
            return "image"
        hint = f"{Path(node.metadata.get('source_file', '')).name}, {node.metadata.get('section', '')}"
        if kind == "table":
            hint += " (Markdown table)"
        node.summary = _llm_summarise(node.content, context_hint=hint)
        return kind
    # combine node: re-combine from (already-fixed) children summaries
    label = node.metadata.get("section") or node.name
    node.summary = _llm_combine_summaries([c.summary for c in node.children], label=label)
    return nt


def repair(case2, progress=None, dry_run=False, regen_images=True, done=None, record=None):
    """Regenerate truncated/forced summaries. Phase 1 = figures (vision model),
    Phase 2 = text/tables/combines (gpt-oss), deepest-first."""
    # the orchestrator for the actual fixing, it splits the work into figures first then everything else, processes the rest deepest-first so a parent re-combines only after its children are already fixed, skips anything already done for crash-resume, and records each fix as it lands, or in dry-run mode just counts what it would do without calling any model
    done = done if done is not None else set()
    images = [n for (n, d) in case2 if n.node_type == "chunk"
              and n.metadata.get("kind") == "image"]
    others = sorted([(n, d) for (n, d) in case2
                     if not (n.node_type == "chunk" and n.metadata.get("kind") == "image")],
                    key=lambda nd: -nd[1])           # deepest first
    counts = {"image": 0, "text": 0, "table": 0, "combine": 0, "skipped": 0}

    if dry_run:
        counts["image"] = len([n for n in images if n.node_id not in done]) if regen_images else 0
        counts["skipped"] = 0 if regen_images else len([n for n in images if n.node_id not in done])
        for n, _ in others:
            if n.node_id in done:
                continue
            counts["combine" if n.node_type != "chunk" else n.metadata.get("kind", "text")] += 1
        return counts

    if not regen_images:
        counts["skipped"] += len([n for n in images if n.node_id not in done])
        images = []

    if images and progress:
        progress.set_phase(f"1/2 figures [{VISION_MODEL}]")
        _warm(VISION_MODEL)
    for n in images:
        if n.node_id in done:
            continue
        t0 = time.time()
        st = _regen_node(n)
        counts["image" if st == "image" else "skipped"] += 1
        done.add(n.node_id)
        if record:
            record(n)
        if progress:
            progress.step(f"figure: {n.name}", time.time() - t0, n.summary)

    if others and progress:
        progress.set_phase(f"2/2 text+combine [{SUMMARY_MODEL}]")
        _warm(SUMMARY_MODEL)
    for n, _ in others:
        if n.node_id in done:
            continue
        t0 = time.time()
        st = _regen_node(n)
        key = {"text": "text", "table": "table"}.get(st, "combine")
        counts[key] += 1
        done.add(n.node_id)
        if record:
            record(n)
        if progress:
            progress.step(f"{n.node_type}: {n.name}", time.time() - t0, n.summary)
    return counts


def save_tree(root, tree_file: Path, backup: bool = True):
    # writes the repaired tree back to disk, keeping a one-time backup of the original first, and writing atomically via a temp file so a crash can't leave you with a half-written tree
    if backup and tree_file.exists():
        bak = tree_file.with_suffix(".backup.json")
        if not bak.exists():
            bak.write_bytes(tree_file.read_bytes())
            print(f"Backed up original tree -> {bak}")
    tmp = tree_file.with_suffix(".writing.json")          # atomic: write temp then replace
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(root.to_dict(), f, ensure_ascii=False, indent=2)
    tmp.replace(tree_file)


# --- crash-resume: an append-only log of repaired summaries -----------------
_repair_log_fh = [None]  # holds the open log file handle in a one-element list so the functions below can share and reassign it


def open_repair_log(log_file: Path):
    # opens the append-only repair log, the running record of fixes that lets a re-run pick up where a crash left off
    log_file.parent.mkdir(parents=True, exist_ok=True)
    _repair_log_fh[0] = open(log_file, "a", encoding="utf-8")


def log_fix(node):
    # appends one repaired node to the log and forces it to disk, so even a hard crash right after won't lose that fix
    fh = _repair_log_fh[0]
    if fh is None:
        return
    fh.write(json.dumps({"node_id": node.node_id, "summary": node.summary,
                         "content": node.content}, ensure_ascii=False) + "\n")
    fh.flush()
    try:
        os.fsync(fh.fileno())                              # survive a hard crash
    except Exception:
        pass


def load_repair_log(log_file: Path):
    """Return {node_id: {summary, content}} from a prior run's log (skips a
    half-written final line)."""
    # reads back a previous run's repair log into a lookup by node id, quietly skipping any final line that was only half-written when a crash hit
    fixes = {}
    if log_file.exists():
        with open(log_file, encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                try:
                    d = json.loads(line)
                except Exception:
                    continue
                if d.get("node_id"):
                    fixes[d["node_id"]] = d
    return fixes


def apply_fixes(root, fixes) -> int:
    """Replay logged summaries onto the freshly-loaded tree. Returns count."""
    # replays the fixes from a prior run onto the freshly-loaded tree, so a resume starts from where it stopped instead of redoing completed work, and reports how many it applied
    applied = 0
    def walk(n):
        # recursive worker that copies a logged summary and content onto a node if there's one for it, then recurses
        nonlocal applied
        f = fixes.get(n.node_id)
        if f is not None:
            n.summary = f.get("summary", n.summary)
            if f.get("content") is not None:
                n.content = f["content"]
            applied += 1
        for c in n.children:
            walk(c)
    walk(root)
    return applied


def update_node_cache(root, progress_iter=lambda x: x):
    """Re-save every node to the per-node cache so a future rebuild is consistent."""
    # writes every node back to the per-node cache so the cache agrees with the repaired tree, otherwise a later rebuild could resurrect the old truncated summaries
    nodes = []
    def collect(n):
        # gathers every node in the tree into a flat list for re-saving
        nodes.append(n)
        for c in n.children:
            collect(c)
    collect(root)
    for n in progress_iter(nodes):
        _save_cached_node(n)
    print(f"Re-saved {len(nodes)} node(s) to {NODE_CACHE_DIR}")


def print_report(verbatim_nodes, report_file: Path = None):
    """Print (and optionally save) every short chunk that was retained verbatim."""
    # prints, and optionally saves, the list of short chunks that were left verbatim on purpose, so you can see exactly what wasn't regenerated and confirm those really were too short to summarise
    lines = []
    lines.append("=" * 80)
    lines.append(f"SHORT CHUNKS RETAINED VERBATIM AS THEIR OWN SUMMARY  ({len(verbatim_nodes)})")
    lines.append("(these are fine — too short to summarise; summary == original chunk)")
    lines.append("=" * 80)
    for i, n in enumerate(verbatim_nodes, 1):
        src = n.metadata.get("source_file", "")
        pg = n.metadata.get("page")
        loc = f"  (p{pg})" if pg is not None else ""
        lines.append(f"[{i}] {Path(src).name if src else n.name}{loc}")
        lines.append(f"    {_norm(n.content)}")
    text = "\n".join(lines)
    print(text)
    if report_file is not None:
        report_file.write_text(text, encoding="utf-8")
        print(f"\n(Report also saved to {report_file})")


print("Audit/repair core ready")

Audit/repair core ready


In [ ]:
fixes = load_repair_log(REPAIR_LOG)  # read back any fixes from a previous run, so a crash can resume instead of starting over
applied = apply_fixes(ROOT, fixes)  # replay those fixes onto the freshly-loaded tree
done = set(fixes)  # the set of node ids already repaired, which repair will skip this run
if applied:
    print(f"Resuming: replayed {applied} previously-repaired summaries from {REPAIR_LOG.name}\n")  # tell the user we picked up where we left off

cls = classify_tree(ROOT)  # walk the tree and bucket every node into complete, verbatim, skipped, or needs-regenerating
todo = [(n, d) for (n, d) in cls["case2"] if n.node_id not in done]  # the regenerate list minus anything already done on a prior run
print("Scanned tree:")  # start the summary report of what the scan found
print(f"  complete (left as-is) : {cls['complete']}")  # summaries that look fine and won't be touched
print(f"  verbatim, kept (<= {SKIP_SUMMARY_WORDS}w): {len(cls['verbatim'])}   (fine; listed in the report)")  # short chunks intentionally left as-is
print(f"  skipped (placeholders): {cls['skipped']}")  # failure placeholders we're leaving alone here
print(f"  flagged to regenerate : {len(cls['case2'])}   (truncated + long-verbatim + all tables)")  # everything the audit wants rebuilt
if cls["reasons"]:
    print("  flagged by:", dict(cls["reasons"]))  # the breakdown of why things were flagged, e.g. how many hit each token cap
print(f"  already done (resume) : {len(done)}")  # how many of those were handled on a previous run
print(f"  REMAINING this run    : {len(todo)}")  # what's actually left to do now
preview = repair(todo, progress=None, dry_run=True, regen_images=REGEN_IMAGES, done=done)  # a dry pass that counts the remaining work by type without calling any model
print("  remaining breakdown   :", {k: v for k, v in preview.items() if v})  # show that breakdown, hiding the zero buckets

if DRY_RUN:
    print("\nDRY_RUN=True -> not regenerating or writing anything.")  # classification-only mode; stop here, change nothing
elif not todo:
    # nothing left to fix, but still write the tree out and optionally refresh the cache so everything's consistent
    print("\nNothing left to regenerate.")
    save_tree(ROOT, TREE_FILE, backup=BACKUP_TREE)  # save the tree, keeping a one-time backup of the original
    if UPDATE_NODE_CACHE:
        from tqdm.auto import tqdm
        update_node_cache(ROOT, progress_iter=lambda xs: tqdm(xs, desc="Rewriting node cache", unit="node"))  # re-save every node so the cache matches the tree
else:
    # logging and a progress bar 
    open_repair_log(REPAIR_LOG)  # open the append-only log so each fix is recorded as it happens
    prog = Progress(len(todo), desc="Repairing summaries")  # one progress bar over the remaining work
    _ck = {"n": 0}  # mutable counter for the checkpoint logic below
    def _record(node):
        # called after each fix; logs it for crash-resume and periodically snapshots the whole tree
        log_fix(node)  # fsync this fix to the log straight away
        _ck["n"] += 1
        if CHECKPOINT_EVERY and _ck["n"] % CHECKPOINT_EVERY == 0:
            save_tree(ROOT, TREE_FILE, backup=BACKUP_TREE)   # periodic full snapshot
    counts = repair(todo, progress=prog, dry_run=False, regen_images=REGEN_IMAGES,
                    done=done, record=_record)  # do the actual regeneration, figures first then text and combines deepest-first
    prog.close()  # shut the bar down and finalise its total
    print("Regenerated this run:", {k: v for k, v in counts.items() if v})  # report what got rebuilt, hiding zero buckets

    # 3) persist final tree + node cache
    save_tree(ROOT, TREE_FILE, backup=BACKUP_TREE)  # write the fully-repaired tree out
    if UPDATE_NODE_CACHE:
        from tqdm.auto import tqdm
        update_node_cache(ROOT, progress_iter=lambda xs: tqdm(xs, desc="Rewriting node cache", unit="node"))  # refresh the per-node cache to match

print()
print_report(cls["verbatim"], report_file=REPORT_FILE)  # finally, print and save the list of short chunks left verbatim

/opt/homebrew/Cellar/jupyterlab/4.5.7_1/libexec/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Scanned tree:
  complete (left as-is) : 83336
  verbatim, kept (<= 8w): 9217   (fine; listed in the report)
  skipped (placeholders): 6936
  flagged to regenerate : 30665   (truncated + long-verbatim + all tables)
  flagged by: {'no end punctuation': 21012, 'table -> regenerate (all)': 9260, 'verbatim >8w -> re-summarise': 322, '~320 tokens': 71}
  already done (resume) : 0
  REMAINING this run    : 30665
  remaining breakdown   : {'text': 10052, 'table': 9260, 'combine': 11353}


2/2 text+combine [gpt-oss:120b]:   7%|██▉                                           | 1999/30665 [2:26:57<28:19:30,  3.56s/fix, ETA 35:07:26 | 0.23/s | last 3.5s]

Backed up original tree -> tree_cache/corpus_tree.backup.json


2/2 text+combine [gpt-oss:120b]:  29%|█████████████▏                               | 9028/30665 [10:28:59<24:21:25,  4.05s/fix, ETA 25:07:28 | 0.24/s | last 3.3s]

...

[waiting for Ollama] ReadTimeout: timed out — retrying...
[waiting for Ollama] ConnectionError: Failed to connect to Ollama. Please check that Ollama is downloaded, running and accessible. https://ollama.com/download — retrying...
[waiting for Ollama] ConnectionError: Failed to connect to Ollama. Please check that Ollama is downloaded, running and accessible. https://ollama.com/download — retrying...


2/2 text+combine [gpt-oss:120b]: 100%|███████████████████████████████████████████████████| 30665/30665 [67:03:26<00:00,  7.87s/fix, ETA 0:00 | 0.13/s | last 6.2s]


Regenerated this run: {'text': 10052, 'table': 9260, 'combine': 11353}


Rewriting node cache: 100%|███████████████████████████████████████████████████████████████████████████████████████████| 130154/130154 [01:06<00:00, 1943.57node/s]

Re-saved 130154 node(s) to tree_cache/nodes

SHORT CHUNKS RETAINED VERBATIM AS THEIR OWN SUMMARY  (9217)
(these are fine — too short to summarise; summary == original chunk)
[1] Amendment 1 WGS Validation Report.pdf  (p19)
    OCT_010303 OCT_010524 OCT_010633 PANX_1248 PANX_1309 PANX_1390 PANX_1391
[2] Amendment 1 WGS Validation Report.pdf  (p36)
    Jun 20, 2025 Medical Director Signature:_____________________________________________ Date:__________________
[3] Amendment 1 WGS Validation Report.pdf  (p37)
    2025-06-20 - 7:10:37 PM GMT
[4] Targeted Sequencing - CHARM Panel Validation Report v1.0.pdf  (p2)
    Page **1** of **31**
[5] Targeted Sequencing - CHARM Panel Validation Report v1.0.pdf  (p3)
    Page **2** of **31**
[6] Targeted Sequencing - CHARM Panel Validation Report v1.0.pdf  (p5)
    Page **4** of **31**
[7] Targeted Sequencing - CHARM Panel Validation Report v1.0.pdf  (p6)
    Page **5** of **31**
[8] Targeted Sequencing - CHARM Panel Validation Report v1.0.pdf  (p7)
 